23070521135 - Sharvayu Zade 

In [7]:
from pathlib import Path

import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Find the dataset from the current folder or one of its parent folders
possible_paths = [Path.cwd() / "creditcard.csv"]
possible_paths += [parent / "Practical 10" / "creditcard.csv" for parent in Path.cwd().parents]
data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("creditcard.csv was not found. Place it in the Practical 10 folder.")

data = pd.read_csv(data_path)

X = data.drop(columns="Class")
y = data["Class"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features using only the training data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Apply SMOTE only to the training data to avoid test-set leakage
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print(f"Dataset shape: {data.shape}")
print(f"Fraud transactions before SMOTE: {y.sum()} ({y.mean() * 100:.2f}%)")
print(f"Training class counts after SMOTE: {y_train.value_counts().to_dict()}")

Dataset shape: (284807, 31)
Fraud transactions before SMOTE: 492 (0.17%)
Training class counts after SMOTE: {0: 227451, 1: 227451}


In [8]:
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

# Build a simple deep neural network
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=2048,
    verbose=1
)

# Convert probabilities to class predictions
probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=["Genuine", "Fraud"], zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

Epoch 1/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8678 - loss: 0.3505 - val_accuracy: 0.9061 - val_loss: 0.1738
Epoch 2/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9625 - loss: 0.1027 - val_accuracy: 0.9448 - val_loss: 0.1178
Epoch 3/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9711 - loss: 0.0760 - val_accuracy: 0.9572 - val_loss: 0.0911
Epoch 4/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9765 - loss: 0.0611 - val_accuracy: 0.9700 - val_loss: 0.0704
Epoch 5/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9810 - loss: 0.0506 - val_accuracy: 0.9809 - val_loss: 0.0546
Epoch 6/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9847 - loss: 0.0427 - val_accuracy: 0.9851 - val_loss: 0.0476
Epoch 7/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9871 - loss: 0.0370 - val_accuracy: 0.9902 - val_loss: 0.0361
Epoch 8/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9892 - loss: 0.0317 - val_accuracy: 0.